In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# Settings
# ============================================================
data_dir = Path(".")
file_pattern = "PEM*.csv"
idle_threshold_mA = 2.0
save_csv = True

# ============================================================
# Helpers
# ============================================================
def load_pem_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    required_cols = ["pc_time", "elapsed_s", "bus_V", "current_mA", "power_mW", "shunt_mV"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing required columns: {missing}")

    df["pc_time"] = pd.to_datetime(df["pc_time"], errors="coerce")

    numeric_cols = ["elapsed_s", "bus_V", "current_mA", "power_mW", "shunt_mV"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["pc_time", "elapsed_s", "bus_V", "current_mA"]).copy()
    df = df.sort_values("pc_time").reset_index(drop=True)

    df["V_V"] = df["bus_V"]
    df["I_mA"] = df["current_mA"]
    df["I_A"] = df["I_mA"] / 1000.0

    return df


def classify_mode(i_mA, idle_threshold_mA):
    if i_mA > idle_threshold_mA:
        return "charge"
    elif i_mA < -idle_threshold_mA:
        return "discharge"
    else:
        return "idle"


def split_into_segments(df: pd.DataFrame, idle_threshold_mA: float) -> pd.DataFrame:
    df = df.copy()
    df["mode"] = df["I_mA"].apply(lambda x: classify_mode(x, idle_threshold_mA))

    active = df[df["mode"] != "idle"].copy()
    if active.empty:
        return active

    new_seg = pd.Series(False, index=active.index)

    # new segment if mode changes
    new_seg |= active["mode"].ne(active["mode"].shift(1))

    # new segment if elapsed time resets
    new_seg |= active["elapsed_s"].diff().fillna(0) < -1

    # new segment if absolute wall-clock gap is large
    dt_pc = active["pc_time"].diff().dt.total_seconds().fillna(0)
    new_seg |= dt_pc > 30

    active["segment_local_id"] = new_seg.cumsum()

    return active


def integrate_segment(seg: pd.DataFrame) -> dict:
    seg = seg.sort_values("pc_time").copy()

    mode = seg["mode"].iloc[0]
    current_mag_A = seg["I_A"].abs()
    power_calc_W = seg["V_V"] * current_mag_A

    dt_s = seg["pc_time"].diff().dt.total_seconds().fillna(0)
    bad_dt = (dt_s < 0) | (dt_s > 60)
    elapsed_dt = seg["elapsed_s"].diff().fillna(0)
    dt_s = dt_s.mask(bad_dt, elapsed_dt).clip(lower=0)

    energy_J = ((power_calc_W + power_calc_W.shift(1)) / 2 * dt_s).fillna(0).sum()
    charge_C = ((current_mag_A + current_mag_A.shift(1)) / 2 * dt_s).fillna(0).sum()

    pmax_idx = power_calc_W.idxmax()

    return {
        "mode": mode,
        "start_time": seg["pc_time"].min(),
        "end_time": seg["pc_time"].max(),
        "duration_s": dt_s.sum(),
        "samples": len(seg),
        "V_start_V": seg["V_V"].iloc[0],
        "V_end_V": seg["V_V"].iloc[-1],
        "V_min_V": seg["V_V"].min(),
        "V_max_V": seg["V_V"].max(),
        "I_mean_mA": current_mag_A.mean() * 1000.0,
        "I_max_mA": current_mag_A.max() * 1000.0,
        "P_mean_mW": power_calc_W.mean() * 1000.0,
        "P_max_mW": power_calc_W.max() * 1000.0,
        "V_at_Pmax_V": seg.loc[pmax_idx, "V_V"],
        "I_at_Pmax_mA": current_mag_A.loc[pmax_idx] * 1000.0,
        "Energy_J": energy_J,
        "Energy_Wh": energy_J / 3600.0,
        "Energy_mWh": energy_J / 3.6,
        "Charge_C": charge_C,
        "Charge_mAh": charge_C / 3.6,
    }


# ============================================================
# Load and segment all files
# ============================================================
files = sorted(data_dir.glob(file_pattern))
if not files:
    raise FileNotFoundError(f"No files matching {file_pattern} found in {data_dir.resolve()}")

all_segments = []

for path in files:
    try:
        df = load_pem_csv(path)
        active = split_into_segments(df, idle_threshold_mA)

        if active.empty:
            print(f"{path.name}: no active segments found")
            continue

        for seg_id, seg in active.groupby("segment_local_id", sort=True):
            metrics = integrate_segment(seg)
            metrics["file"] = path.name
            metrics["segment_local_id"] = int(seg_id)
            all_segments.append(metrics)

    except Exception as e:
        print(f"Error in {path.name}: {e}")

segments_df = pd.DataFrame(all_segments)

if segments_df.empty:
    raise ValueError("No charge/discharge segments found in any files.")

segments_df = segments_df.sort_values("start_time").reset_index(drop=True)
segments_df["segment_global_id"] = range(1, len(segments_df) + 1)

# Reorder columns
segment_cols = [
    "segment_global_id", "file", "segment_local_id", "mode",
    "start_time", "end_time", "duration_s", "samples",
    "V_start_V", "V_end_V", "V_min_V", "V_max_V",
    "I_mean_mA", "I_max_mA",
    "P_mean_mW", "P_max_mW", "V_at_Pmax_V", "I_at_Pmax_mA",
    "Energy_J", "Energy_Wh", "Energy_mWh", "Charge_C", "Charge_mAh"
]
segments_df = segments_df[segment_cols]

# ============================================================
# Pair charge segment with next discharge segment
# ============================================================
cycle_rows = []
pending_charge = None
cycle_id = 1

for _, row in segments_df.iterrows():
    if row["mode"] == "charge":
        pending_charge = row

    elif row["mode"] == "discharge":
        if pending_charge is not None:
            charge_energy = pending_charge["Energy_mWh"]
            discharge_energy = row["Energy_mWh"]

            cycle_rows.append({
                "cycle_id": cycle_id,

                "charge_segment_global_id": pending_charge["segment_global_id"],
                "charge_file": pending_charge["file"],
                "charge_start_time": pending_charge["start_time"],
                "charge_end_time": pending_charge["end_time"],
                "charge_duration_s": pending_charge["duration_s"],
                "charge_energy_mWh": charge_energy,
                "charge_mAh": pending_charge["Charge_mAh"],
                "charge_Pmax_mW": pending_charge["P_max_mW"],
                "charge_Imax_mA": pending_charge["I_max_mA"],

                "discharge_segment_global_id": row["segment_global_id"],
                "discharge_file": row["file"],
                "discharge_start_time": row["start_time"],
                "discharge_end_time": row["end_time"],
                "discharge_duration_s": row["duration_s"],
                "discharge_energy_mWh": discharge_energy,
                "discharge_mAh": row["Charge_mAh"],
                "discharge_Pmax_mW": row["P_max_mW"],
                "discharge_Imax_mA": row["I_max_mA"],

                "energy_out_in_pct": 100 * discharge_energy / charge_energy if charge_energy > 0 else pd.NA,
                "mAh_out_in_pct": 100 * row["Charge_mAh"] / pending_charge["Charge_mAh"] if pending_charge["Charge_mAh"] > 0 else pd.NA,
            })

            cycle_id += 1
            pending_charge = None

cycles_df = pd.DataFrame(cycle_rows)

# ============================================================
# Print results
# ============================================================
print("\n================ ALL ACTIVE SEGMENTS ================\n")
print(segments_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

print("\n================ PAIRED CHARGE / DISCHARGE CYCLES ================\n")
if cycles_df.empty:
    print("No charge-discharge pairs found.")
else:
    print(cycles_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# ============================================================
# Save CSV files
# ============================================================
if save_csv:
    segments_df.to_csv("PEM_all_active_segments.csv", index=False)
    cycles_df.to_csv("PEM_charge_discharge_cycles.csv", index=False)

    print("\nSaved:")
    print("  PEM_all_active_segments.csv")
    print("  PEM_charge_discharge_cycles.csv")


================ ALL ACTIVE SEGMENTS ================

 segment_global_id                                                file  segment_local_id      mode              start_time                end_time  duration_s  samples  V_start_V  V_end_V  V_min_V  V_max_V  I_mean_mA  I_max_mA  P_mean_mW  P_max_mW  V_at_Pmax_V  I_at_Pmax_mA  Energy_J  Energy_Wh  Energy_mWh  Charge_C  Charge_mAh
                 1                                    PEM_charging.csv                 1    charge 2026-04-22 14:20:19.763 2026-04-22 14:56:14.976   2155.2130     2152     0.4137   1.6413   0.4137   1.6600    75.1910  122.4000   122.2638  152.6143       1.6575       92.0750  263.5430     0.0732     73.2064  162.0307     45.0085
                 2           PEM_Charge_ina226_log_20260422_142001.csv                 1    charge 2026-04-22 14:20:19.763 2026-04-22 14:56:14.976   2155.2130     2152     0.4137   1.6413   0.4137   1.6600    75.1910  122.4000   122.2638  152.6143       1.6575       92.0750  263.5430